# Vitessce Visualization Workflow

This notebook demonstrates how to use `cvh-client` with [Vitessce](https://vitessce.io/) to:

1. Create a Vitessce visualization in a CVH workspace
2. Retrieve it from the API
3. View it locally using the Vitessce widget
4. Modify the configuration and push the update back

## Setup

In [ ]:
from cvh_client import CVHClient
from vitessce import VitessceConfig, Component as cm, CoordinationType as ct

## Authenticate

In [ ]:
client = CVHClient.from_login(
    base_url="https://api.example.com",       # replace with your CVH API URL
    domain="your-tenant.auth0.com",            # replace with your Auth0 domain
    client_id="your-client-id",                # replace with your Auth0 client ID
    audience="https://api.example.com",        # replace with your API audience
)

## Pick a workspace

In [ ]:
workspaces = client.list_workspaces()
for ws in workspaces.items:
    print(f"{ws.name} — {ws.uuid}")

workspace_uuid = workspaces.items[0].uuid

## 1. Create a visualization using the Codeluppi et al. osmFISH example

This uses the [Codeluppi et al., Nature Methods 2018](https://vitessce.io/#?dataset=codeluppi-2018) example config from Vitessce, which visualizes spatial organization of the somatosensory cortex revealed by osmFISH.

In [ ]:
conf = {
    "name": "Codeluppi et al., Nature Methods 2018",
    "description": "Spatial organization of the somatosensory cortex revealed by osmFISH",
    "version": "1.0.15",
    "initStrategy": "auto",
    "datasets": [
        {
            "uid": "codeluppi",
            "name": "Codeluppi",
            "files": [
                {
                    "fileType": "obsSegmentations.json",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.segmentations.json",
                },
                {
                    "fileType": "obsLocations.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.csv",
                    "options": {"obsIndex": "cell_id", "obsLocations": ["X", "Y"]},
                    "coordinationValues": {"obsType": "cell"},
                },
                {
                    "fileType": "obsEmbedding.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.csv",
                    "options": {"obsIndex": "cell_id", "obsEmbedding": ["PCA_1", "PCA_2"]},
                    "coordinationValues": {"obsType": "cell", "embeddingType": "PCA"},
                },
                {
                    "fileType": "obsEmbedding.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.csv",
                    "options": {"obsIndex": "cell_id", "obsEmbedding": ["TSNE_1", "TSNE_2"]},
                    "coordinationValues": {"obsType": "cell", "embeddingType": "t-SNE"},
                },
                {
                    "fileType": "obsSets.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.csv",
                    "options": {
                        "obsIndex": "cell_id",
                        "obsSets": [{"name": "Cell Type", "column": ["Cluster", "Subcluster"]}],
                    },
                    "coordinationValues": {"obsType": "cell"},
                },
                {
                    "fileType": "obsLocations.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.molecules.csv",
                    "options": {"obsIndex": "molecule_id", "obsLocations": ["X", "Y"]},
                    "coordinationValues": {"obsType": "molecule"},
                },
                {
                    "fileType": "obsLabels.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.molecules.csv",
                    "options": {"obsIndex": "molecule_id", "obsLabels": "Gene"},
                    "coordinationValues": {"obsType": "molecule"},
                },
                {
                    "fileType": "obsFeatureMatrix.csv",
                    "url": "https://data-1.vitessce.io/0.0.33/main/codeluppi-2018/codeluppi_2018_nature_methods.cells.matrix.csv",
                    "coordinationValues": {"obsType": "cell", "featureType": "gene", "featureValueType": "expression"},
                },
                {
                    "fileType": "image.raster.json",
                    "options": {
                        "schemaVersion": "0.0.2",
                        "images": [
                            {
                                "name": "Image",
                                "url": "https://vitessce-data.storage.googleapis.com/0.0.31/master_release/linnarsson/linnarsson.images.zarr",
                                "type": "zarr",
                                "metadata": {
                                    "dimensions": [
                                        {"field": "channel", "type": "nominal", "values": ["polyT", "nuclei"]},
                                        {"field": "y", "type": "quantitative", "values": None},
                                        {"field": "x", "type": "quantitative", "values": None},
                                    ],
                                    "isPyramid": True,
                                    "transform": {"translate": {"y": 0, "x": 0}, "scale": 1},
                                },
                            }
                        ],
                    },
                },
            ],
        }
    ],
    "coordinationSpace": {
        "embeddingZoom": {"PCA": 0, "TSNE": 0.75},
        "embeddingType": {"PCA": "PCA", "TSNE": "t-SNE"},
        "spatialZoom": {"A": -5.5},
        "spatialTargetX": {"A": 16000},
        "spatialTargetY": {"A": 20000},
        "spatialSegmentationLayer": {"A": {"opacity": 1, "radius": 0, "visible": True, "stroked": False}},
        "spatialPointLayer": {"A": {"opacity": 1, "radius": 20, "visible": True}},
    },
    "layout": [
        {"component": "description", "props": {"description": "Codeluppi et al., Nature Methods 2018: Spatial organization of the somatosensory cortex revealed by osmFISH"}, "x": 0, "y": 0, "w": 2, "h": 1},
        {"component": "layerController", "coordinationScopes": {"spatialSegmentationLayer": "A", "spatialPointLayer": "A"}, "x": 0, "y": 1, "w": 2, "h": 4},
        {"component": "status", "x": 0, "y": 5, "w": 2, "h": 1},
        {"component": "spatial", "coordinationScopes": {"spatialZoom": "A", "spatialTargetX": "A", "spatialTargetY": "A", "spatialSegmentationLayer": "A", "spatialPointLayer": "A"}, "props": {"channelNamesVisible": True}, "x": 2, "y": 0, "w": 4, "h": 4},
        {"component": "featureList", "x": 9, "y": 0, "w": 3, "h": 2},
        {"component": "obsSets", "x": 9, "y": 3, "w": 3, "h": 2},
        {"component": "heatmap", "props": {"transpose": True}, "x": 2, "y": 4, "w": 5, "h": 2},
        {"component": "obsSetFeatureValueDistribution", "x": 7, "y": 4, "w": 5, "h": 2},
        {"component": "scatterplot", "coordinationScopes": {"embeddingType": "PCA", "embeddingZoom": "PCA"}, "x": 6, "y": 0, "w": 3, "h": 2},
        {"component": "scatterplot", "coordinationScopes": {"embeddingType": "TSNE", "embeddingZoom": "TSNE"}, "x": 6, "y": 2, "w": 3, "h": 2},
    ],
}
conf

In [ ]:
# Create the visualization in CVH and attach the config
viz = client.create_visualization(
    workspace_uuid,
    name="Codeluppi et al., Nature Methods 2018",
    description="Spatial organization of the somatosensory cortex revealed by osmFISH",
    tool="vitessce",
)
client.update_visualization(viz.uuid, conf=conf)
print(f"Created visualization: {viz.uuid}")

## 2. Retrieve the visualization from CVH

In [ ]:
fetched_viz = client.get_visualization(viz.uuid)
print(f"Name: {fetched_viz.name}")
print(f"Tool: {fetched_viz.tool}")
print(f"UUID: {fetched_viz.uuid}")

## 3. View the visualization with Vitessce

Load the stored config back into a `VitessceConfig` object and display it as a widget.

> **Note:** The Vitessce Jupyter widget requires `pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ cvh-client[vitessce]` for full widget support.

In [ ]:
# Load the config from CVH into a VitessceConfig and display it
vc_fetched = VitessceConfig.from_dict(fetched_viz.conf)
vc_fetched.widget()

## 4. Modify the config and update the visualization

Modify the config retrieved from CVH (e.g. adjust the spatial zoom), then push the update back.

In [ ]:
# Modify the config — zoom in on the spatial view
updated_conf = fetched_viz.conf.copy()
updated_conf["coordinationSpace"]["spatialZoom"]["A"] = -3.0
updated_conf["coordinationSpace"]["spatialTargetX"]["A"] = 18000
updated_conf["coordinationSpace"]["spatialTargetY"]["A"] = 18000
updated_conf

In [ ]:
# Push the updated config back to CVH
client.update_visualization(viz.uuid, conf=updated_conf)